<a href="https://colab.research.google.com/github/mihikag-tech/UTD-Deep-Dive-AI-Project/blob/test/XGB_troubleshoot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pickle

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv("/content/Combined_dataset_model.csv")
df = pd.get_dummies(df, columns=["biome"], dtype=int)
df = df.drop(columns=['Unnamed: 0'])
print(df.columns)


Index(['county', 'land_area', 'tc_goal', 'treecanopy', 'tc_gap', 'priority_i',
       'pctpocnorm', 'pctpovnorm', 'unemplnorm', 'dep_perc', 'depratnorm',
       'health_nor', 'temp_norm', 'tes', 'tesctyscor', 'rank', 'rankgrpsz',
       'Mean_Temp', 'Median_Temp', 'STD_Temp', 'Min_Temp', 'Max_Temp',
       'Mean_Rain', 'Median_Rain', 'STD_Rain', 'Min_Rain', 'Max_Rain',
       'biome_Desert', 'biome_Forest', 'biome_Grassland'],
      dtype='object')


In [ ]:
features = ['land_area', 'tc_goal', 'treecanopy', 'tc_gap',
       'priority_i', 'pctpocnorm', 'pctpovnorm', 'unemplnorm', 'dep_perc',
       'depratnorm', 'tes', 'tesctyscor', 'rank',
       'rankgrpsz', 'Mean_Temp', 'Median_Temp', 'STD_Temp', 'Min_Temp',
       'Max_Temp', 'Mean_Rain', 'Median_Rain', 'STD_Rain', 'Min_Rain',
       'Max_Rain', 'biome_Desert', 'biome_Forest', 'biome_Grassland']
target = ['health_nor']
X_df = df[features]
y_df = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size = 0.2, random_state = 42)

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
}

grid_search = GridSearchCV(
    XGBRegressor(random_state = 42, objective='reg:squarederror'),
    param_grid,
    cv = 5,
    scoring = 'r2'
)

grid_search.fit(X_train, y_train)

print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))
print("Test acc    :", round(grid_search.best_estimator_.score(X_test, y_test), 4))

Best params : {'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 300}
Best CV acc : 0.6718
Test acc    : 0.674


In [ ]:
best_model = grid_search.best_params_
best_model

{'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 300}

In [ ]:
"""
model = XGBRegressor(learning_rate = 0.2, max_depth = 4, n_estimators = 300)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)
"""

model = pickle.load(open('xgb_model_pickle', 'rb'))
model_pred = model.predict(X_test)
print("R2 score:", r2_score(y_test, model_pred))



R2 score: 0.6739763617515564


In [ ]:
# Define what each solution actually changes, and by how much
solution_effects = {
    "street_trees": {
        "treecanopy": 50,       # adds 0.75 percentage points of canopy
    },

    "community_garden": {
        "treecanopy": 0.25,
    },
}

In [ ]:
import pandas as pd
feature_cols = features

def estimate_intervention_impact(model, area_row, solution_name, solution_effects, feature_cols):
    """
    area_row: a pandas Series (one row of the dataset) representing current conditions
    Returns: baseline_prediction, new_prediction, estimated_impact
    """
    # Baseline: predict on the area as it currently is
    baseline_features = area_row[feature_cols].values.reshape(1, -1)
    baseline_pred = model.predict(baseline_features)[0]

    # Copy the row and apply the intervention's effects
    modified_row = area_row.copy()
    for feature, delta in solution_effects[solution_name].items():
        modified_row[feature] = modified_row[feature] + delta
        # Clip to realistic bounds, e.g. percentages can't exceed 100 or go below 0
        if "pct" in feature:
            modified_row[feature] = max(0, min(100, modified_row[feature]))

    modified_features = modified_row[feature_cols].values.reshape(1, -1)
    new_pred = model.predict(modified_features)[0]

    impact = new_pred - baseline_pred
    return baseline_pred, new_pred, impact

In [ ]:


solution_costs = {
    "street_trees": 100,  # Example cost
    "community_garden": 50, # Example cost
}

results = []
for idx, area_row in df.iterrows():  # df = your priority areas from Task 1
    for solution_name in solution_effects:
        baseline, new_pred, impact = estimate_intervention_impact(
            model, area_row, solution_name, solution_effects, feature_cols
        )
        results.append({
            "area_id": area_row["county"],
            "solution": solution_name,
            "baseline_prediction": baseline,
            "predicted_outcome": new_pred,
            "estimated_impact": impact,
            "cost": solution_costs[solution_name],
        })

impact_df = pd.DataFrame(results)

In [ ]:
impact_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32022 entries, 0 to 32021
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   area_id              32022 non-null  object 
 1   solution             32022 non-null  object 
 2   baseline_prediction  32022 non-null  float32
 3   predicted_outcome    32022 non-null  float32
 4   estimated_impact     32022 non-null  float32
 5   cost                 32022 non-null  int64  
dtypes: float32(3), int64(1), object(2)
memory usage: 1.1+ MB


In [ ]:
impact_df.head(10)

,area_id,solution,baseline_prediction,predicted_outcome,estimated_impact,cost
0,Anderson County,street_trees,0.471488,0.635889,0.164401,100
1,Anderson County,community_garden,0.471488,0.635889,0.164401,50
2,Anderson County,street_trees,0.573419,0.620756,0.047337,100
3,Anderson County,community_garden,0.573419,0.620756,0.047337,50
4,Anderson County,street_trees,0.600754,0.741084,0.140331,100
5,Anderson County,community_garden,0.600754,0.741084,0.140331,50
6,Anderson County,street_trees,0.644553,0.657248,0.012694,100
7,Anderson County,community_garden,0.644553,0.657248,0.012694,50
8,Anderson County,street_trees,0.820066,0.911432,0.091366,100
9,Anderson County,community_garden,0.820066,0.911432,0.091366,50
